In [20]:
import os
import transformers
from helper_functions import *
import json





batch_path = "eval_p3"
transform_lct ="/work/eauten2s/ec_criteria_struct/lct"

# Model
model_id =  "meta-llama/Meta-Llama-3-70B-Instruct"
model_name = "Llama-3-70B-Instruct"

# N Shots
n_shot = 5 # liefert genau die Anzahl Beispiele (study, label)

# Input/ Output
study_path = f"{transform_lct}/input/lct_txt_half/"

# Model
model_id =  "meta-llama/Meta-Llama-3-70B-Instruct"
model_name = "Llama-3-70B-Instruct"
n_shot=2
# Input/ Output
study_path = f"{transform_lct}/input/lct_txt_half/"
output_path = f"{transform_lct}/evaluate/{batch_path}/model_output/{model_name}_{n_shot}_shot/output/"
os.makedirs(output_path, exist_ok=True)


anfang = 0
ende = 300
study_files = os.listdir(study_path)[anfang:ende]


# Load Model Description
model_desc = read_text_file(f"{transform_lct}/input/prompt/prompt_p3.txt")

def read_matching_p3_files(study_folder, n_shot):
    study_filenames = []
    study_contents = []
    label_filenames = []
    label_contents = []

    loaded_files = 0
    for file_name in os.listdir(study_folder):
        if file_name.endswith(".txt"):
            study_filenames.append(file_name)
            study_file_path = os.path.join(study_folder, file_name)
            with open(study_file_path, 'r', encoding='utf-8') as file:
                study_contents.append(file.read())

            label_file_name = file_name.replace(".txt", "_p3.json")
            label_file_path = os.path.join(study_folder, label_file_name)
            print(label_file_path)
            if os.path.exists(label_file_path):
                label_filenames.append(label_file_name)
                with open(label_file_path, 'r', encoding='utf-8') as file:
                    label_contents.append(file.read())
            else:
                label_filenames.append(None)
                label_contents.append(None)
            loaded_files += 1
            if loaded_files == n_shot * 2:
                break

    return study_filenames, study_contents, label_filenames, label_contents


# Load n-shot Data
study_folder = f"{transform_lct}/input/n_shot_files_p3"

study_filenames, study_contents, label_filenames, label_contents = read_matching_p3_files(study_folder, n_shot)
studies = dict(zip(study_filenames, study_contents))
labels = dict(zip(label_filenames, label_contents))
messages = []

command = "Bring the following eligibility criterias in Json format with logical operators and extract entitys:"

messages.append({"role": "system", "content": f"{model_desc}"})

for i in range(n_shot):
    messages.append({"role": "user", "content": f"{command} {studies[study_filenames[i]]}"})
    messages.append({"role": "assistant", "content": labels[label_filenames[i]]})





first_call = True

for file in study_files:

    print(file)
    test_file = read_text_file(study_path+file)
    print(test_file)

    if first_call:
        messages.append({"role": "user", "content": f"{command} {test_file}"})
        first_call = False
    else:
        messages[-1] = {"role": "user", "content": f"{command} {test_file}"}




In [21]:
messages